> 在线讲义：[第 9 章 · encoder-only 的 BERT：遮词再猜，以及为什么现在几乎都是 decoder-only](https://wikieden.github.io/beyond-transformer/lessons/09-bert.html) · 本书《超越 Transformer》

# 第 9 章 · encoder-only 的 BERT：遮词再猜，以及为什么现在几乎都是 decoder-only

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/09-bert.html) · 仓库里 `beyond-transformer/notes/lessons/09-bert.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 两种 mask | 全 1 / 下三角 | `masks(5)` | 每行 5 个 vs 1–5 个 |
| 第二步 遮词 | 15%、80 / 10 / 10 | `mlm_corrupt(SENT)`、`corrupt_stats()` | my → cat，to → [MASK]；12.19% / 1.51% / 1.49% |
| 第二步 loss | 选中位置取平均 | `mlm_loss([0.6, 0.3])` | 0.8574 |
| 第四步 信号密度 | 15% vs 每个位置 | `signal_per_sequence(512)` | 77 vs 511 |

loss 只在被选中的位置算；每个 token 带来的训练信号 MLM 只有 15%，而 decoder-only 每个位置都算。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
SENT = ["[CLS]", "my", "dog", "is", "hairy", "and", "he", "likes", "to", "play", "fetch", "[SEP]"]
VOCAB = ["the", "cat", "apple", "run", "blue", "happy"]

**函数 `masks`**：双向：全 1；因果：下三角（第 i 个只看 0..i）。

In [ ]:
def masks(n=5):
    """双向：全 1；因果：下三角（第 i 个只看 0..i）。"""
    return np.ones((n, n), int), np.tril(np.ones((n, n), int))

**函数 `mlm_corrupt`**：选中 15% 的位置（不选 [CLS]/[SEP]）；选中的 80% 换 [MASK]、10% 换随机词、10% 保持原样。

In [ ]:
def mlm_corrupt(tokens, seed=3, rate=0.15):
    """选中 15% 的位置（不选 [CLS]/[SEP]）；选中的 80% 换 [MASK]、10% 换随机词、10% 保持原样。"""
    rng = np.random.default_rng(seed)
    cand = [i for i, t in enumerate(tokens) if not t.startswith("[")]
    k = max(1, round(rate * len(cand)))
    chosen = sorted(rng.choice(cand, k, replace=False).tolist())
    out, how = list(tokens), {}
    for i in chosen:
        r = rng.uniform()
        if r < 0.8:
            out[i], how[i] = "[MASK]", "换成 [MASK]"
        elif r < 0.9:
            out[i], how[i] = str(rng.choice(VOCAB)), "换成随机词"
        else:
            how[i] = "保持原样"
    return out, chosen, how

**函数 `corrupt_stats`**：大量 token 上统计：被选中、换 [MASK]、换随机、不变各占多少。

In [ ]:
def corrupt_stats(n=100_000, seed=0):
    """大量 token 上统计：被选中、换 [MASK]、换随机、不变各占多少。"""
    rng = np.random.default_rng(seed)
    sel = rng.uniform(size=n) < 0.15
    r = rng.uniform(size=n)
    return {"被选中": sel.mean(), "换 [MASK]": (sel & (r < 0.8)).mean(), "换随机词": (sel & (r >= 0.8) & (r < 0.9)).mean(), "不变": (sel & (r >= 0.9)).mean()}

**函数 `mlm_loss`**：只在被选中的位置取 −log p(原词) 的平均。

In [ ]:
def mlm_loss(probs_at_chosen):
    """只在被选中的位置取 −log p(原词) 的平均。"""
    return float(np.mean([-np.log(p) for p in probs_at_chosen]))

**函数 `signal_per_sequence`**：一条长 n 的序列能给多少个“猜词”的训练信号。

In [ ]:
def signal_per_sequence(n=512):
    """一条长 n 的序列能给多少个“猜词”的训练信号。"""
    return {"MLM（15%）": int(round(0.15 * n)), "decoder-only（每个位置）": n - 1}

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 双向 vs 因果")
bi, ca = masks()
print(f"  双向：每行 {bi.sum(1).tolist()} 个能看的位置\n  因果：每行 {ca.sum(1).tolist()} 个")
print("\n第二步 MLM 遮词")
out, chosen, how = mlm_corrupt(SENT)
print(f"  原句：{' '.join(SENT)}")
print(f"  遮后：{' '.join(out)}")
for i in chosen:
    print(f"  位置 {i}（{SENT[i]}）：{how[i]}")
print("  10 万个 token 的统计：" + "，".join(f"{k} {v:.2%}" for k, v in corrupt_stats().items()))

In [ ]:
print("\n第三步 loss 只在被选中的位置算")
print(f"  两个选中位置的原词概率 0.6、0.3：MLM loss = (−log 0.6 − log 0.3)/2 = {mlm_loss([0.6, 0.3]):.4f}")

In [ ]:
print("\n第四步 为什么现在都用 decoder-only")
for k, v in signal_per_sequence().items():
    print(f"  序列长 512，{k}：{v} 个训练信号")

## 3. 代码里才看得到的

- **“保持原样”的位置也要算 loss**：输入看起来没变，但它在被选中的集合里，模型必须猜出它自己。这正是让每个位置都“靠得住”的那 10%。
- **小句子按比例四舍五入**：10 个普通词的 15% 是 1.5 个，这里取 2；真实实现是每个位置独立以 15% 的概率选，一条短句可能一个都没选到。

## 4. 自己改一改

- 把 `mlm_corrupt` 的 seed 换几个，看选中的位置和处理方式怎么变。
- 把 `signal_per_sequence` 的 n 改成 128、2048，比例变吗？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 阅读理解看两边，写作只看前面。
> 看两边不能猜自己，答案就在输入里。
> 完形填空十五分，八成换遮罩、一成换随机、一成不动；只在选中处算 loss。
> 句首 CLS 当总结，SEP 隔两句，三种嵌入相加。
> 每个位置都出信号、天生会生成、还能 KV cache，所以大模型都选 decoder-only。